# SOCCAT — Replicate Results from Hugging Face Hub

Loads each fine-tuned model from the Hub and runs inference on the corresponding NLI pair data to reproduce the evaluation tables reported in the paper — **no retraining required**.

**To replicate:**
1. Request the NLI pair data files by contacting the authors
2. Place them in a local folder and update `DATA_ROOT` in the *Configuration* cell
3. Run all cells in order

**Models:** All 8 fine-tuned models are publicly available at [huggingface.co/selsar](https://huggingface.co/selsar).

**Data:** The annotated NLI pair CSVs are available from the authors upon request. The annotation conversion script (`convert_annotations.py`) is included in the replication package for full transparency.

## 1. Installation

In [ ]:
%%capture
!pip install transformers datasets accelerate openpyxl scikit-learn -U

## 2. Imports

In [ ]:
import csv
import json
import os

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, average_precision_score, cohen_kappa_score,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from transformers import DataCollatorWithPadding
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    Trainer, TrainingArguments,
)

## 3. Configuration

| Variable | Description |
|---|---|
| `DATA_ROOT` | Directory containing the NLI pair CSV files |
| `OUT_DIR` | Output directory for predictions and metric tables |
| `MANIFEST` | Path to `categories.json` |
| `CATEGORY` | Restrict to one category by name, or `None` for all |
| `BATCH_SIZE` | Eval batch size — reduce if you hit OOM |

In [ ]:
# ── Update these paths before running ────────────────────────────────────
DATA_ROOT  = "nli_pairs_by_category/"   # folder with nli_dataset_*.csv files
OUT_DIR    = "replication_output/"       # where results will be saved
MANIFEST   = "categories.json"           # path to categories.json
CATEGORY   = None                         # e.g. "real_estate_ownership", or None for all
BATCH_SIZE = 32                           # reduce to 8 if running on CPU
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)

with open(MANIFEST) as f:
    manifest = json.load(f)

if CATEGORY:
    manifest = [c for c in manifest if c["name"] == CATEGORY]
    if not manifest:
        raise ValueError(f"Category '{CATEGORY}' not found in {MANIFEST}.")

print(f"Categories to evaluate: {[c['name'] for c in manifest]}")

## 4. Data Loading & Dataset Class

In [ ]:
def load_data(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path, engine="openpyxl")
    else:
        with open(path, "r", encoding="utf-8", newline="") as f:
            dialect = csv.Sniffer().sniff(f.read(4096), delimiters=",;\t|")
        df = pd.read_csv(path, delimiter=dialect.delimiter)

    df["sentence_id"] = df["sentence_id"].astype(int)
    df["nli_label"]   = df["nli_label"].astype(int)

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        if "year" not in df.columns:
            df["year"] = df["date"].dt.year.astype("Int64")

    if "decade" not in df.columns:
        if "date" in df.columns:
            mask = df["date"].notna()
            df["decade"] = pd.NA
            if mask.any():
                df.loc[mask, "decade"] = (df.loc[mask, "date"].dt.year // 10 * 10).astype("Int64")
        elif "year" in df.columns:
            df["decade"] = (pd.to_numeric(df["year"], errors="coerce") // 10 * 10).astype("Int64")
    return df


class NLIDataset:
    def __init__(self, df, tokenizer, max_length=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row["premise"], row["hypothesis"],
            truncation=True,
            max_length=self.max_length,
            )
        enc["labels"] = int(row["nli_label"])
        return enc

## 5. Metrics

In [ ]:
def softmax(logits):
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def compute_metrics(y_true, prob_entail):
    """label 0 = entailment (positive), 1 = not_entailment."""
    y_pred   = np.where(prob_entail > 0.5, 0, 1)
    pos_mask = (y_true == 0).astype(int)
    out = dict(
        accuracy          = accuracy_score(y_true, y_pred),
        precision_binary  = precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        recall_binary     = recall_score(y_true, y_pred, pos_label=0),
        f1_binary         = f1_score(y_true, y_pred, pos_label=0),
        precision_micro   = precision_score(y_true, y_pred, average="micro", zero_division=0),
        recall_micro      = recall_score(y_true, y_pred, average="micro"),
        f1_micro          = f1_score(y_true, y_pred, average="micro"),
        f1_macro          = f1_score(y_true, y_pred, average="macro"),
        cohen_kappa       = cohen_kappa_score(y_true, y_pred),
        prevalence        = float(pos_mask.mean()),
    )
    try:    out["roc_auc"] = roc_auc_score(pos_mask, prob_entail)
    except: out["roc_auc"] = np.nan
    try:    out["pr_auc"]  = average_precision_score(pos_mask, prob_entail)
    except: out["pr_auc"]  = np.nan
    return out


def per_group_metrics(df, prob_entail, group_col):
    tmp = df.copy()
    tmp["prob_entail"] = prob_entail
    rows = []
    for g, sub in tmp.groupby(group_col, dropna=False):
        yt = sub["nli_label"].astype(int).values
        pe = sub["prob_entail"].values
        m  = compute_metrics(yt, pe)
        m.update({group_col: g, "n_pairs": len(sub), "n_pos_entail": int((yt == 0).sum())})
        rows.append(m)
    return pd.DataFrame(rows)

## 6. Inference — All Categories

Loads each Hub model, runs inference, and saves per-category predictions and metric tables.

In [ ]:
all_results = []
skipped     = []

for cat in manifest:
    name      = cat["name"]
    hub_repo  = cat["hub_repo"]
    data_file = os.path.join(DATA_ROOT, os.path.basename(cat["data"]))
    out_dir   = os.path.join(OUT_DIR, name)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  Category : {cat['display_name']}")
    print(f"  Model    : https://huggingface.co/{hub_repo}")
    print(f"  Data     : {data_file}")

    if not os.path.exists(data_file):
        print(f"  WARNING: data file not found — skipping.")
        skipped.append(name)
        continue

    df        = load_data(data_file)
    tokenizer = AutoTokenizer.from_pretrained(hub_repo, use_fast=True)
    model     = AutoModelForSequenceClassification.from_pretrained(hub_repo)
    dataset   = NLIDataset(df, tokenizer)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=os.path.join(out_dir, "tmp"),
            per_device_eval_batch_size=BATCH_SIZE,
            report_to=[],
        ),
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )

    pred        = trainer.predict(dataset)
    logits      = pred.predictions[0] if isinstance(pred.predictions, (tuple, list)) else pred.predictions
    prob_entail = softmax(logits)[:, 0]
    y_true      = df["nli_label"].astype(int).values
    y_pred      = np.where(prob_entail > 0.5, 0, 1)

    df["prob_entail"] = prob_entail
    df["pred_label"]  = y_pred

    overall = compute_metrics(y_true, prob_entail)
    overall["category"] = name

    df.to_csv(os.path.join(out_dir, "predictions.csv"), index=False)
    with open(os.path.join(out_dir, "metrics_overall.json"), "w") as f:
        json.dump(overall, f, indent=2)
    for col in ["hypothesis_label", "outlet", "country", "year", "decade"]:
        if col in df.columns:
            per_group_metrics(df, prob_entail, col).to_csv(
                os.path.join(out_dir, f"metrics_per_{col}.csv"), index=False
            )

    print(f"  F1 (binary): {overall['f1_binary']:.4f} | "
          f"F1 (macro): {overall['f1_macro']:.4f} | "
          f"Accuracy: {overall['accuracy']:.4f}")
    all_results.append(overall)

if skipped:
    print(f"\nSkipped (data not found): {skipped}")

## 7. Summary Table

Collects overall metrics across all evaluated categories into a single table.

In [ ]:
if all_results:
    summary = pd.DataFrame(all_results).set_index("category")
    cols = ["accuracy", "f1_binary", "f1_macro", "f1_micro",
            "precision_binary", "recall_binary", "cohen_kappa", "pr_auc", "roc_auc", "prevalence"]
    cols = [c for c in cols if c in summary.columns]
    summary = summary[cols].round(4)

    print("\n" + "="*60)
    print("REPLICATION SUMMARY — all categories")
    print("="*60)
    print(summary.to_string())

    summary.to_csv(os.path.join(OUT_DIR, "summary_all_categories.csv"))
    with pd.ExcelWriter(os.path.join(OUT_DIR, "summary_all_categories.xlsx")) as writer:
        summary.to_excel(writer, sheet_name="Summary")

    print(f"\nSummary saved to: {OUT_DIR}")
else:
    print("No results to summarise — check that data files exist in DATA_ROOT.")